## ResNET + DistillBERT Multi-Modal Classification

# Imports and Model Setup

In [ ]:

import os
import re
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms, datasets
from transformers import AutoTokenizer, AutoModel
from torchvision.models import resnet50, ResNet50_Weights
    
    # Checking Model Availablility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Paths and Hyper Parameters

In [ ]:
# Model Paths
IMAGE_WEIGHTS = "/home/le.song/Assignment2/ResNET_garbage_best_image_model_Fine_Tuned.pth"
TEXT_WEIGHTS  = "/home/le.song/Assignment2/clean_text_classifier.pth"
FUSION_SAVE_PATH = "/home/le.song/Assignment2/best_fusion_model.pth"

# Data Paths
DATA_ROOT  = "/work/TALC/ensf617_2026w/garbage_data"
TRAIN_PATH = os.path.join(DATA_ROOT, "CVPR_2024_dataset_Train")
VAL_PATH   = os.path.join(DATA_ROOT, "CVPR_2024_dataset_Val")
TEST_PATH  = os.path.join(DATA_ROOT, "CVPR_2024_dataset_Test")

# Hyperparameters
BATCH_SIZE = 16

# Image Encoder

In [ ]:
# Image Encoder
class ImageEncoder(nn.Module):
    def __init__(self, num_classes=4, proj_dim=768, pretrained_weights_path=None, device='cpu'):
        super().__init__()
        # Load ResNet50 backbone
        self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        # Replace classifier with identity
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Load pretrained fine-tuned weights
        if pretrained_weights_path is not None:
            state_dict = torch.load(pretrained_weights_path, map_location=device)
            
            # Remove classifier keys if necessary (only keep feature extractor)
            backbone_state = {k.replace('feature_extractor.', ''): v 
                            for k, v in state_dict.items() 
                            if k.startswith('feature_extractor.')}
            
            self.backbone.load_state_dict(backbone_state, strict=False)
            print("Loaded fine-tuned ResNet50 weights into ImageEncoder")
        
        # Flatten features and project to same dimension as text
        self.flatten = nn.Flatten()
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU()
        )
    # Forward
    def forward(self, x):
        features = self.backbone(x)
        features = self.flatten(features)  
        return self.projection(features)  

# Multi-Modal Model Definition

In [ ]:
class MultiModalModel(nn.Module):
        def __init__(self, image_encoder, text_base_model, num_classes=4):
            super().__init__()
            #Setup for learnable modalidy trust(variable weights for image/text depending on situation)
            self.modality_logit = nn.Parameter(torch.tensor(0.0))
            self.image_encoder = image_encoder
            self.text_encoder = text_base_model
                
            # Both are now 768
            self.text_norm = nn.LayerNorm(text_base_model.config.hidden_size)
                
            fusion_input_dim = 768 + 768 # 1536 total
                
            self.fusion_head = nn.Sequential(
                nn.Linear(1536, 256),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(256, num_classes)
            )

        def forward(self, images, input_ids, attention_mask):
            # Image branch
            img_emb = self.image_encoder(images) 
            
            # Text branch
            text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden = text_outputs.last_hidden_state
            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            text_emb = (last_hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
            
            # Normalize text embedding to match the image projection's scale
            text_emb = self.text_norm(text_emb)
            # Modality Dropout to prevent label memorization since backbones are permanently frozen
            if self.training:
                # 10% chance to drop the text signal
                mask = (torch.rand(text_emb.size(0), 1, device=text_emb.device) > 0.1).float()
                text_emb = text_emb * mask
            #Convert logits to weight
            alpha = torch.sigmoid(self.modality_logit)
            #Weight Scaled image/text embeddings on modality trust
            img_emb  = alpha * img_emb
            text_emb = (1 - alpha) * text_emb
            #Concatenation after all processing is complete
            x = torch.cat([img_emb, text_emb], dim=1)
            return self.fusion_head(x)

# Data-Loading

In [ ]:
# Data Loading

def get_text_data(path):
    texts, labels = [], []
    class_folders = sorted([f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f)) and not f.startswith(".")])
    label_map = {name: i for i, name in enumerate(class_folders)}
    for name in class_folders:
        class_path = os.path.join(path, name)
        for fname in sorted(os.listdir(class_path)):
            text = os.path.splitext(fname)[0].replace('_', ' ')
            text = re.sub(r'\d+', '', text)
            texts.append(text)
            labels.append(label_map[name])
    return texts, labels

class MultiModalDataset(Dataset):
    def __init__(self, image_ds, texts, labels, tokenizer, max_len=128):
        self.image_ds = image_ds
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        img, _ = self.image_ds[idx]
        encoding = self.tokenizer(str(self.texts[idx]), padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "image": img,
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Pre-training/validation setups and model/weights loading + backbones freezing

In [ ]:
# Setup Image Transformers
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Components
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
img_ds_train = datasets.ImageFolder(TRAIN_PATH, transform=test_transform)
img_ds_val   = datasets.ImageFolder(VAL_PATH, transform=test_transform)
img_ds_test  = datasets.ImageFolder(TEST_PATH, transform=test_transform)

txt_train, lbl_train = get_text_data(TRAIN_PATH)
txt_val, lbl_val     = get_text_data(VAL_PATH)
txt_test, lbl_test   = get_text_data(TEST_PATH)

train_loader = DataLoader(MultiModalDataset(img_ds_train, txt_train, lbl_train, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(MultiModalDataset(img_ds_val, txt_val, lbl_val, tokenizer), batch_size=BATCH_SIZE)
test_loader  = DataLoader(MultiModalDataset(img_ds_test, txt_test, lbl_test, tokenizer), batch_size=BATCH_SIZE)

# Loading Image Weights
encoder = ImageEncoder(num_classes=4,proj_dim=768,pretrained_weights_path=IMAGE_WEIGHTS,device=device).to(device)
# Loading Text Model
base_text_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
# Loading Text Weights
txt_state = torch.load(TEXT_WEIGHTS, map_location=device)
# Filter state_dict for text base model
text_base_state = {k.replace('base_model.', ''): v for k, v in txt_state.items() if k.startswith('base_model.')}
base_text_model.load_state_dict(text_base_state)

# Assemble Fusion Model
multimodal_model = MultiModalModel(encoder, base_text_model).to(device)
print("Fusion Model assembled and weights loaded from unimodal training.")

#Both Backbones Frozen
for param in multimodal_model.image_encoder.backbone.parameters():
    param.requires_grad = False
for param in multimodal_model.text_encoder.parameters():
    param.requires_grad = False
#Also freeze image projection
for param in multimodal_model.image_encoder.projection.parameters():
    param.requires_grad = False

# Training/Evaluation Loop

In [ ]:
# Training / Evaluation

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, multimodal_model.parameters()),
    lr=1e-5
)

EPOCHS = 20

train_losses, val_losses = [], []
best_val_loss = float('inf')
for epoch in range(EPOCHS):

    # Training 
    multimodal_model.train()
    running_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)              # torch.Tensor
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = multimodal_model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(multimodal_model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    multimodal_model.eval()
    val_loss_total = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = multimodal_model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss_total += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss_total / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}")
    alpha_value = torch.sigmoid(multimodal_model.modality_logit).item()
    print(f"Image modality trust: {alpha_value:.3f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(multimodal_model.state_dict(), FUSION_SAVE_PATH)
        print(f"Saved best model to {FUSION_SAVE_PATH}")

# Loss Chart

In [ ]:
# Plotting Multimodal Loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Multimodal Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig('/home/le.song/Assignment2/multimodal_loss_curve1.png')
plt.show()

# Testing/Inference with Metrics, CM, Logs, misclassification & best model tracking

In [ ]:
def evaluate_multimodal_errors(model, test_loader, device, log_file="/home/le.song/Assignment2/mm_misclassified_final.txt"):
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    model.eval()
    
    all_preds = []
    all_labels = []
    incorrect_samples = []
    image_paths = test_loader.dataset.image_ds.samples

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images, input_ids, attention_mask)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Identify misclassifications for logging
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    global_idx = batch_idx * test_loader.batch_size + i
                    path, _ = image_paths[global_idx]
                    error_entry = f"Path: {path} | Pred: {preds[i].item()} | True: {labels[i].item()}"
                    incorrect_samples.append(error_entry)
        # 1. Print Standard Metrics
    print("\n" + "="*30)
    print("MULTIMODAL FINAL PERFORMANCE")
    print("="*30)
    report = classification_report(all_labels, all_preds, target_names=test_loader.dataset.image_ds.classes)
    print(report)

    # 2. Generate and Save Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_loader.dataset.image_ds.classes)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Multimodal Confusion Matrix")
    plt.savefig('/home/le.song/Assignment2/mm_confusion_matrix.png')
    print(f"Confusion Matrix saved to /home/le.song/Assignment2/mm_confusion_matrix.png")

    # 3. Log errors
    with open(log_file, "w") as f:
        for line in incorrect_samples:
            f.write(line + "\n")

    # Calculate accuracy for return
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"\nOverall accuracy: {accuracy:.4f}")

    return accuracy

# Loading Best MM_Model and Running Inference

In [ ]:
# Load best weights before final eval
multimodal_model.load_state_dict(torch.load(FUSION_SAVE_PATH, map_location=device))
mm_accuracy = evaluate_multimodal_errors(multimodal_model, test_loader, device)
print(f"Final Multimodal Accuracy: {mm_accuracy * 100:.22f}%")